# Lab 08: Best practices (reference solution)

**Time:** about 2 h 30 min.

**Checklist questions this lab gives you evidence for:**<br><span style="color:#d93025"><b>6.1 Why avoid UDFs?</b></span><br><span style="color:#d93025"><b>6.3 Broadcast variables</b></span><br><span style="color:#d93025"><b>6.4 Star schema benefit</b></span><br><span style="color:#d93025"><b>6.5 lit() in loops</b></span><br><span style="color:#d93025"><b>6.6 Data lineage</b></span><br><span style="color:#d93025"><b>6.7 Built-in functions over Python logic</b></span><br><span style="color:#d93025"><b>6.8 Hardcoded values</b></span><br><span style="color:#d93025"><b>6.9 Late data in streaming</b></span>

**Goal:** measure what a Python UDF and a `withColumn` loop cost, refactor a messy job, test your code, and handle late data in a stream. The last section is practice with array functions and text cleaning.

## Data information
A small online shop. The setup cell below creates three tables in `data/` (seeded, so everyone gets the same rows):

**orders** (`data/orders.parquet`, 1,000,000 rows, January to March 2024)
* *order_id*, *customer_id* (empty for guest checkout), *product_id*, *quantity* (1 to 5)
* *amount*: quantity × product price, in dollars
* *status*: paid, shipped, cancelled or returned; *payment_method*: card, cash or wallet
* *order_ts*, *order_date*: when the order was placed
* *tracking_code*: a 32-character code

**customers** (`data/customers.csv`, 10,000 rows): *customer_id*, *name*, *country*, *signup_date*, *segment* (retail or business). Customer 1 is "Big Reseller Ltd", who places 30% of all orders.

**products** (`data/products.csv`, 200 rows): *product_id*, *name*, *category*, *price*

## How to work
* Write your answer in the code cell under each question, then run the check cell below it: ✅ correct, ❌ try again.
* When a question says *guess first*, write your guess as a comment before you run anything.
* Keep the Spark UI open (the setup cell prints its address).
* Stuck for more than 10 minutes? Read the documentation list at the top, then the reference solution.


## Documentation
Everything you need is on these pages. The questions do not repeat them: read, try things out in a cell, and find the answers yourself.

* [PySpark guide: UDFs and UDTFs](https://spark.apache.org/docs/latest/api/python/user_guide/udfandudtf.html)
* [Arrow Python UDFs](https://spark.apache.org/docs/latest/api/python/tutorial/sql/arrow_python_udf.html)
* [functions.pandas_udf](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.pandas_udf.html)
* [DataFrame.withColumn](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumn.html)
* [DataFrame.withColumns](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumns.html)
* [Functions in PySpark guide](https://spark.apache.org/docs/latest/api/python/user_guide/dataprep.html)
* [functions.initcap](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.initcap.html)
* [functions.broadcast](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.broadcast.html)
* [Testing PySpark](https://spark.apache.org/docs/latest/api/python/getting_started/testing_pyspark.html)
* [pyspark.testing.assertDataFrameEqual](https://spark.apache.org/docs/latest/api/python/reference/pyspark.testing.html)
* [Dynamic partition pruning (configuration)](https://spark.apache.org/docs/latest/sql-performance-tuning.html)
* [functions.weekday](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.weekday.html)
* [RDD guide: broadcast variables](https://spark.apache.org/docs/latest/rdd-programming-guide.html#broadcast-variables)
* [SparkContext.broadcast](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.SparkContext.broadcast.html)
* [Generic file source options: hidden metadata](https://spark.apache.org/docs/latest/sql-data-sources-generic-options.html)
* [Handling late data and watermarking](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html#handling-late-data-and-watermarking)
* [DataStreamWriter](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamWriter.html)
* [functions.filter / transform / aggregate](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.aggregate.html)
* [functions.collect_list](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.collect_list.html)
* [functions.regexp_replace](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.regexp_replace.html)
* [functions.substring](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.substring.html)

## Setup
Run the cell below once (its code is hidden: click it to expand). It creates the data and starts Spark.

In [ ]:
# Setup: run this cell once. It creates the shop data in data/ and starts Spark.
# Nothing to solve here, and you do not need to read it.
import contextlib, hashlib, io, json, os, re, shutil, sys, time, urllib.request
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this kernel's Python
from pyspark.sql import SparkSession, Window, functions as F

# ---------------------------------------------------------------- the shop data ---------
# A small online shop, generated from seeded hash functions: every machine gets the same rows.
COUNTRIES = ["Uzbekistan", "Germany", "USA", "India", "Brazil", "Japan"]
CATEGORIES = ["books", "toys", "garden", "sports", "kitchen", "beauty", "music", "office"]
N_CUSTOMERS, N_PRODUCTS, N_ORDERS = 10_000, 200, 1_000_000

def _u(col, salt):
    """Pseudo-random number in [0, 1) computed from a column (same result on every machine)."""
    return F.pmod(F.xxhash64(col, F.lit(salt)), F.lit(1_000_000)) / 1_000_000.0

def _pick(u, options):
    return F.element_at(F.array(*[F.lit(o) for o in options]), (F.floor(u * len(options)) + 1).cast("int"))

def make_customers(spark):
    c = spark.range(1, N_CUSTOMERS + 1).withColumnRenamed("id", "customer_id")
    return c.select(
        "customer_id",
        F.when(F.col("customer_id") == 1, "Big Reseller Ltd")
         .otherwise(F.concat(F.lit("Customer "), F.col("customer_id").cast("string"))).alias("name"),
        _pick(_u(F.col("customer_id"), 1), COUNTRIES).alias("country"),
        F.date_add(F.lit("2022-01-01").cast("date"), F.floor(_u(F.col("customer_id"), 2) * 730).cast("int")).alias("signup_date"),
        F.when((F.col("customer_id") == 1) | (_u(F.col("customer_id"), 3) < 0.2), "business").otherwise("retail").alias("segment"),
    )

def make_products(spark):
    p = spark.range(1, N_PRODUCTS + 1).withColumnRenamed("id", "product_id")
    return p.select(
        "product_id",
        F.concat(F.lit("Product "), F.col("product_id").cast("string")).alias("name"),
        F.element_at(F.array(*[F.lit(c) for c in CATEGORIES]), (F.col("product_id") % 8 + 1).cast("int")).alias("category"),
        F.round((F.col("product_id") * 37) % 195 + 5.99, 2).alias("price"),
    )

def make_orders(spark, n=N_ORDERS, dirty=False):
    i = F.col("id")
    customer = (F.when(_u(i, 11) < 0.02, F.lit(None).cast("long"))                    # guest checkout: no customer
                 .when(_u(i, 10) < 0.30, F.lit(1).cast("long"))                       # the big reseller: 30% of orders
                 .otherwise((F.floor(_u(i, 12) * (N_CUSTOMERS - 1)) + 2).cast("long")))
    quantity = F.when(customer == 1, F.lit(5)).otherwise(F.floor(_u(i, 13) * 5) + 1).cast("int")   # the reseller always orders in bulk
    product = (F.floor(_u(i, 14) * N_PRODUCTS) + 1).cast("int")
    ts = F.lit("2024-01-01 00:00:00").cast("timestamp") + F.make_interval(secs=F.floor(_u(i, 15) * 91 * 86400).cast("int"))
    u = _u(i, 17)
    status = F.when(u < 0.55, "paid").when(u < 0.80, "shipped").when(u < 0.92, "cancelled").otherwise("returned")
    v = _u(i, 18)
    payment = F.when(v < 0.60, "card").when(v < 0.85, "cash").otherwise("wallet")
    df = spark.range(0, n, 1, 4).select(
        (i + 1).alias("order_id"), customer.alias("customer_id"), product.alias("product_id"), quantity.alias("quantity"),
        F.round(quantity * (((product * 37) % 195) + 5.99), 2).alias("amount"),
        status.alias("status"), payment.alias("payment_method"), ts.alias("order_ts"), F.md5(i.cast("string")).alias("tracking_code"))
    df = df.withColumn("order_date", F.to_date("order_ts"))
    if dirty:    # the monthly files of the final project contain mistakes on purpose
        df = df.withColumn("amount", F.when(_u(F.col("order_id"), 19) < 0.008, -F.col("amount")).otherwise(F.col("amount")))
        df = df.withColumn("quantity", F.when(_u(F.col("order_id"), 20) < 0.006, 0).otherwise(F.col("quantity")))
        df = df.unionByName(df.filter(_u(F.col("order_id"), 21) < 0.005))              # exact duplicates
    return df.select("order_id", "customer_id", "product_id", "quantity", "amount", "status",
                     "payment_method", "order_ts", "order_date", "tracking_code")

def _create_data(spark):
    os.makedirs("data/monthly", exist_ok=True)
    make_customers(spark).toPandas().to_csv("data/customers.csv", index=False)
    make_products(spark).toPandas().to_csv("data/products.csv", index=False)
    make_orders(spark).repartition(4).write.mode("overwrite").parquet("data/orders.parquet")
    dirty = make_orders(spark, dirty=True)
    wrong = _u(F.col("order_id"), 22) < 0.003          # a few orders land in the wrong month's file
    month = F.month("order_date")
    file_month = F.when(wrong, month % 3 + 1).otherwise(month)
    for m in (1, 2, 3):
        (dirty.filter(file_month == m).coalesce(1).write.mode("overwrite")
              .parquet(f"data/monthly/orders_2024-{m:02d}.parquet"))

# ---------------------------------------------------------------- the Spark session ----
def new_session(master="local[4]", driver_memory="2g", delta=False, avro=False, aqe=False, conf=None):
    builder = (SparkSession.builder.master(master).appName("day08_lab")
               .config("spark.driver.memory", driver_memory)
               .config("spark.ui.showConsoleProgress", "false")
               .config("spark.sql.adaptive.enabled", str(aqe).lower())   # off = exact job and stage counts
               .config("spark.sql.session.timeZone", "UTC")                # same dates and hours for everyone
               .config("spark.sql.warehouse.dir", os.path.abspath("spark-warehouse"))
               # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this.
               .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false"))
    for k, v in (conf or {}).items():
        builder = builder.config(k, v)
    if delta:
        from delta import configure_spark_with_delta_pip
        builder = (builder.config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
                          .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
        builder = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.spark:spark-avro_2.13:4.2.0"] if avro else None)
    elif avro:
        builder = builder.config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.2.0")
    s = builder.getOrCreate()
    s.sparkContext.setLogLevel("ERROR")
    return s

for _d in ("spark-warehouse", "metastore_db"):
    shutil.rmtree(_d, ignore_errors=True)               # a clean catalog on every start
spark = new_session()
if not all(os.path.exists(f) for f in ("data/orders.parquet", "data/customers.csv", "data/products.csv", "data/monthly/orders_2024-03.parquet")):
    print("creating the shop data (about 30 seconds, once)...")
    _create_data(spark)
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

try:
    from delta.tables import DeltaTable   # for the Delta labs
except ImportError:
    pass

# ---------------------------------------------------------------- helpers ----------------
def _tracker():
    return spark.sparkContext.statusTracker()

def _run_group(action):
    sc = spark.sparkContext
    group = f"g{time.time_ns()}"
    sc.setJobGroup(group, group)
    try:
        action()
    finally:
        sc.setLocalProperty("spark.jobGroup.id", None)
    return [_tracker().getJobInfo(j) for j in sorted(_tracker().getJobIdsForGroup(group))]

def jobs_run(action):
    """How many Spark jobs calling action() starts."""
    return len(_run_group(action))

def stages_run(action):
    """How many stages those jobs had in total."""
    return sum(len(j.stageIds) for j in _run_group(action))

def last_stage_tasks(action):
    """Number of tasks in the last stage action() ran."""
    jobs = _run_group(action)
    return _tracker().getStageInfo(max(s for j in jobs for s in j.stageIds)).numTasks

def _api(path):
    sc = spark.sparkContext
    return json.load(urllib.request.urlopen(f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}{path}"))

api = _api   # read anything from the Spark UI's REST API, e.g. api("/executors")

def _stage_rows(action):
    jobs = _run_group(action)
    ids = {s for j in jobs for s in j.stageIds}
    return [s for s in _api("/stages?length=2000") if s["stageId"] in ids]

def shuffle_bytes(action):
    """Total shuffle bytes written by the stages action() ran."""
    return sum(s["shuffleWriteBytes"] for s in _stage_rows(action))

def spill_bytes(action):
    """(memory, disk) bytes spilled by the stages action() ran."""
    rows = _stage_rows(action)
    return sum(s["memoryBytesSpilled"] for s in rows), sum(s["diskBytesSpilled"] for s in rows)

def skew_ratio(action):
    """Slowest task / median task, in the stage with the most tasks that action() ran."""
    rows = _stage_rows(action)
    st = max(rows, key=lambda s: s["numTasks"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=5000")
    times = sorted(t["taskMetrics"]["executorRunTime"] for t in tasks if t.get("taskMetrics"))
    return round(times[-1] / max(times[len(times) // 2], 1), 1)

def failed_attempts():
    """Failed task attempts in the most recent failed stage (from the Spark UI's REST API)."""
    failed = _api("/stages?status=failed")
    if not failed:
        return 0
    st = max(failed, key=lambda s: s["stageId"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=500")
    return sum(1 for t in tasks if t["status"] == "FAILED")

def sql_metric(action, metric):
    """Value of one scan metric (for example 'number of files read') of the query action() runs."""
    desc = f"q{time.time_ns()}"
    spark.sparkContext.setJobDescription(desc)
    try:
        action()
    finally:
        spark.sparkContext.setJobDescription(None)
    for ex in _api("/sql?details=true&planDescription=false&length=2000"):
        if ex.get("description") == desc:
            vals = [m["value"] for n in ex["nodes"] for m in n["metrics"] if m["name"] == metric]
            return vals[0] if vals else None

def noop(df):
    """Run the whole query but throw the result away."""
    df.write.format("noop").mode("overwrite").save()

def plan(df, mode="formatted"):
    """explain() output as a string."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode)
    return buf.getvalue()

def count_nodes(df, *names):
    """How often each operator name appears in the simple plan."""
    text = plan(df, "simple")
    return {n: len(re.findall(r"\b" + re.escape(n) + r"\b", text)) for n in names}

def optimized(df):
    """The optimized logical plan as a string."""
    return df._jdf.queryExecution().optimizedPlan().toString()

def operators(df):
    """Operator names of the physical plan tree, top to bottom."""
    tree = plan(df).split("\n\n")[0].splitlines()[1:]
    return [re.sub(r"\s*\(\d+\)$", "", line).strip(" +-*:") for line in tree]

def rules_fired(df):
    """Names of the Catalyst rules that changed this query's plan."""
    qe = df._jdf.queryExecution(); qe.executedPlan()
    it, fired = qe.tracker().rules().iterator(), set()
    while it.hasNext():
        kv = it.next()
        if kv._2().numEffectiveInvocations() > 0:
            fired.add(kv._1().split(".")[-1].split("$")[-1])
    return fired

def estimate(df):
    """(sizeInBytes, rowCount or None) the optimizer estimates for df."""
    stats = df._jdf.queryExecution().optimizedPlan().stats()
    return int(str(stats.sizeInBytes())), (int(str(stats.rowCount().get())) if stats.rowCount().isDefined() else None)

def partition_sizes(df):
    """Rows per partition, largest first."""
    return sorted((r["count"] for r in df.groupBy(F.spark_partition_id().alias("p")).count().collect()), reverse=True)

def timeit(action):
    start = time.perf_counter(); action(); return round(time.perf_counter() - start, 2)

def folder_mb(path):
    files = [os.path.join(d, f) for d, _, fs in os.walk(path) for f in fs] if os.path.isdir(path) else [path]
    return round(sum(os.path.getsize(f) for f in files) / 1e6, 1)

# ---------------------------------------------------------------- checks -----------------
_EXPECTED = {
 "8.1": "3102678fdfd273a9",
 "8.5": "0906e522a1a6e42a",
 "8.6": "8094a8bd0f98d2ea",
 "8.7": "4339021aa34621f3",
 "8.8": "135c7fb84f2bc14c",
 "8.9": "27b17a75c4a99c86",
 "8.10": "6d9fc0acbccf7bb4"
}
_RECORD = os.environ.get("LAB_RECORD")

def _norm(v):
    if isinstance(v, bool) or v is None: return v
    if isinstance(v, int): return v
    if isinstance(v, float): return round(v, 2)
    if isinstance(v, str): return v.strip()
    if isinstance(v, dict): return {str(k): _norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
    if isinstance(v, (list, tuple)): return [_norm(x) for x in v]
    if isinstance(v, (set, frozenset)): return sorted((_norm(x) for x in v), key=json.dumps)
    if hasattr(v, "item"): return _norm(v.item())
    if type(v).__name__ == "Decimal": return round(float(v), 2)
    raise TypeError(f"check() needs plain Python values, got {type(v).__name__}")

def check(task, value):
    normal = _norm(value)
    if _RECORD:
        rec = json.load(open(_RECORD)) if os.path.exists(_RECORD) else {}
        rec[task] = normal; json.dump(rec, open(_RECORD, "w"))
    ok = hashlib.sha256(json.dumps(normal).encode()).hexdigest()[:16] == _EXPECTED.get(task)
    print(f"✅ {task} correct" if ok else f"❌ {task} not yet: check your code and the question again")

def check_true(task, condition, hint=""):
    print(f"✅ {task} correct" if condition else f"❌ {task} not yet. {hint}")


Load the tables the way Lab 01 taught you (given):

In [ ]:
orders = spark.read.parquet("data/orders.parquet")
customers = (spark.read.option("header", True)
             .schema("customer_id long, name string, country string, signup_date date, segment string")
             .csv("data/customers.csv"))
products = (spark.read.option("header", True)
            .schema("product_id long, name string, category string, price double")
            .csv("data/products.csv"))
for name, df in [("orders", orders), ("customers", customers), ("products", products)]:
    df.createOrReplaceTempView(name)   # so SQL can use the tables too

## A. UDFs and loops

**8.1 What does a UDF cost?** Add 12% VAT to every *amount* (`round(amount * 1.12, 2)`) in three ways: built-in functions (*builtin*), a Python UDF (*py_udf*) and a pandas UDF (*pd_udf*), each as `orders.select(...alias("v"))`. Run each with `noop` and time it (*t_builtin*, *t_py*, *t_pd*): in Spark 4.2 a simple UDF is no longer dramatically slower. What it still costs is what Spark can do with the query. Check four things and put them in *result*:
* *sums_equal*: the three sums of *v* are equal (True/False)
* *udf_node*: the plan of the UDF version contains `ArrowEvalPython` or `BatchEvalPython`
* *pushed_builtin*: the *PushedFilters* of `orders.filter(F.col("amount") > 500)` (a list; use `pushed_of`, given)
* *pushed_udf*: the *PushedFilters* of the same filter written with a Python UDF

`result = [sums_equal, udf_node, pushed_builtin, pushed_udf]`

In [ ]:
def pushed_of(df):   # given: the PushedFilters of a plan
    m = re.search(r"PushedFilters: \[(.*?)\]", plan(df))
    return m.group(1).split(", ") if m and m.group(1) else []

import pandas as pd
from pyspark.sql.functions import pandas_udf, udf

py_vat = udf(lambda a: round(a * 1.12, 2), "double")

@pandas_udf("double")
def pd_vat(a: pd.Series) -> pd.Series:
    return (a * 1.12).round(2)

builtin = orders.select(F.round(F.col("amount") * 1.12, 2).alias("v"))
py_udf = orders.select(py_vat("amount").alias("v"))
pd_udf = orders.select(pd_vat("amount").alias("v"))
t_builtin, t_py, t_pd = timeit(lambda: noop(builtin)), timeit(lambda: noop(py_udf)), timeit(lambda: noop(pd_udf))
sums = [round(d.agg(F.sum("v")).first()[0], 2) for d in (builtin, py_udf, pd_udf)]
sums_equal = len(set(sums)) == 1
udf_node = any(count_nodes(py_udf, "ArrowEvalPython", "BatchEvalPython").values())
over_500 = udf(lambda a: a is not None and a > 500, "boolean")
pushed_builtin = pushed_of(orders.filter(F.col("amount") > 500))
pushed_udf = pushed_of(orders.filter(over_500("amount")))
result = [sums_equal, udf_node, sorted(pushed_builtin), pushed_udf]
print("seconds:", t_builtin, t_py, t_pd)
result

In [ ]:
check("8.1", result)

**8.2 Why avoid `withColumn(lit)` in a loop?** On `spark.range(10)`, add 300 columns `c0` to `c299` (`F.lit(i)`) in two ways and time only building the DataFrame (not running it): a loop with `withColumn` (*t_loop*) and one `select("*", *[...])` (*t_select*). *loop_slower* = True if the loop takes more than twice as long.

In [ ]:
base = spark.range(10)

def with_loop():
    df = base
    for i in range(300):
        df = df.withColumn(f"c{i}", F.lit(i))
    return df

t_loop = timeit(with_loop)
t_select = timeit(lambda: base.select("*", *[F.lit(i).alias(f"c{i}") for i in range(300)]))
loop_slower = t_loop > 2 * t_select
t_loop, t_select, loop_slower

In [ ]:
check_true("8.2", loop_slower, "every withColumn re-analyzes the growing plan, so 300 calls are much slower than one select")

## B. Refactor and test a job

**8.3 Refactor `bad_report()` (given).** Write `good_report(config)` that returns the same result without the problems: no Python UDF (use a built-in function), no loop of useless columns, filter before the join, the small table *customers* broadcast, and no hardcoded numbers or countries (take them from *config*). Then compare: the result must be equal (*same*), the plan must have no `EvalPython` node and a `BroadcastHashJoin`, and *good_report* must be faster.

In [ ]:
def bad_report():
    """A slow, messy job (given): revenue per country and payment method for big orders from two countries."""
    clean = F.udf(lambda s: s.capitalize(), "string")
    big = F.udf(lambda a: a is not None and a > 100, "boolean")
    df = orders
    for i in range(40):
        df = df.withColumn(f"flag_{i}", F.lit(i))
    df = df.join(customers.hint("merge"), "customer_id")
    df = df.withColumn("payment", clean("payment_method")).filter(big("amount")).filter(F.col("country").isin(["Germany", "USA"]))
    return df.groupBy("country", "payment").agg(F.round(F.sum("amount"), 2).alias("revenue"))

CONFIG = {"min_amount": 100, "countries": ["Germany", "USA"]}   # no hardcoded values in your version

def good_report(config):
    big = orders.filter(F.col("amount") > config["min_amount"])
    cust = customers.filter(F.col("country").isin(config["countries"]))
    return (big.join(F.broadcast(cust), "customer_id")
               .withColumn("payment", F.initcap("payment_method"))
               .groupBy("country", "payment").agg(F.round(F.sum("amount"), 2).alias("revenue")))

In [ ]:
from pyspark.testing import assertDataFrameEqual
try:
    assertDataFrameEqual(good_report(CONFIG), bad_report())
    same = True
except AssertionError:
    same = False
text = plan(good_report(CONFIG), "simple")
t_bad, t_good = timeit(lambda: noop(bad_report())), timeit(lambda: noop(good_report(CONFIG)))
check_true("8.3", same and "EvalPython" not in text and "BroadcastHashJoin" in text, "same result, no Python UDF node, broadcast join")
check_true("8.3b", t_good < t_bad, "the refactored job should be faster")

**8.4 Test your code.** Write `add_vat(df, rate)`: it returns *df* with a new column *amount_with_vat* = `round(amount * (1 + rate), 2)`. The check cell tests it on a tiny DataFrame with `assertDataFrameEqual`, the way real projects test Spark code.

In [ ]:
def add_vat(df, rate):
    return df.withColumn("amount_with_vat", F.round(F.col("amount") * (1 + rate), 2))

In [ ]:
from pyspark.testing import assertDataFrameEqual
sample = spark.createDataFrame([(1, 100.0), (2, 19.99)], "order_id int, amount double")
expected = spark.createDataFrame([(1, 100.0, 119.0), (2, 19.99, 23.79)], "order_id int, amount double, amount_with_vat double")
try:
    assertDataFrameEqual(add_vat(sample, 0.19), expected)
    ok = True
except Exception as e:
    ok = False
check_true("8.4", ok, "add_vat(sample, 0.19) should equal the expected DataFrame")

## C. Star schema, broadcast variables, lineage

**8.5 What does a star schema give you?** Write *orders* partitioned by *order_date* to `lake8/orders_by_date` and read it as *fact*. *dates* (given) is a small date dimension with the column *dow* (0 = Monday). Join *fact* with `dates.filter("dow = 5")` (Saturdays) on *order_date* and run it with `noop`.
* *pruned*: does the fact scan's *PartitionFilters* contain `dynamicpruning`? (True/False)
* *partitions_read*: the scan metric `number of partitions read` (use `sql_metric(action, name)`, make it an `int`)

In [ ]:
dates = (spark.range(91).select(F.date_add(F.lit("2024-01-01").cast("date"), F.col("id").cast("int")).alias("order_date"))
         .withColumn("dow", F.weekday("order_date")))
shutil.rmtree("lake8", ignore_errors=True)
orders.write.partitionBy("order_date").parquet("lake8/orders_by_date")
fact = spark.read.parquet("lake8/orders_by_date")
q = fact.join(dates.filter("dow = 5"), "order_date")
pruned = bool(re.search(r"PartitionFilters: \[[^\]]*dynamicpruning", plan(q)))
partitions_read = int(str(sql_metric(lambda: noop(q), "number of partitions read")).split()[0])
pruned, partitions_read

In [ ]:
check("8.5", [pruned, partitions_read])

**8.6 Broadcast variable or broadcast join?** Count the orders per product *category* in two ways: with a Python dict `{product_id: category}` sent through `spark.sparkContext.broadcast(...)` and used in a UDF (*by_udf*, dict `{category: count}`), and with a broadcast join with *products* (*by_join*). Time both (*t_udf*, *t_join*). Equal results (*equal*) and the join must be faster.

In [ ]:
lookup = spark.sparkContext.broadcast({r["product_id"]: r["category"] for r in products.collect()})
category_of = F.udf(lambda pid: lookup.value.get(pid), "string")
udf_q = orders.select(category_of("product_id").alias("category")).groupBy("category").count()
join_q = orders.join(F.broadcast(products), "product_id").groupBy("category").count()
t_udf, t_join = timeit(udf_q.collect), timeit(join_q.collect)
by_udf = {r["category"]: r["count"] for r in udf_q.collect()}
by_join = {r["category"]: r["count"] for r in join_q.collect()}
equal = by_udf == by_join
equal, t_udf, t_join

In [ ]:
check("8.6", by_join)
check_true("8.6b", equal and t_join < t_udf, "same result, and the broadcast join should be faster than the UDF")

**8.7 Where did each row come from?** Add the name of the source file to *orders* with the hidden `_metadata` column (`F.col("_metadata.file_name")`) and count the rows per file. Put `[number of files, total rows]` in *result*.

In [ ]:
by_file = orders.select(F.col("_metadata.file_name").alias("file")).groupBy("file").count()
rows = by_file.collect()
result = [len(rows), sum(r["count"] for r in rows)]
result

In [ ]:
check("8.7", result)

## D. Late data in streaming

**8.8 What happens to late events?** Events arrive as JSON files in `stream_in/`. Write `run()`: it reads the folder as a stream (schema `ts timestamp, user string`), sets a 10 minute watermark on *ts*, counts the events per 5 minute `window`, and writes the result in append mode as Parquet to `stream_out` (checkpoint `stream_ckpt`) with `trigger(availableNow=True)`, then waits for it. `drop_events` and `read_out` are given.

Drop four files one after another, call `run()` after each, and save `read_out()`:
* `b1`: minutes 0, 1, 6, 9 → *r1*
* `b2`: minute 31 → *r2*
* `b3`: minutes 7 (late!) and 22 → *r3*
* `b4`: minute 45 → *r4*

Put `[r1, r2, r3, r4]` in *result*. Guess first: when is a window written, and what happens to minute 7?

In [ ]:
import json

def drop_events(name, minutes):
    """Another application drops a file of events into stream_in/ (given)."""
    os.makedirs("stream_in", exist_ok=True)
    with open(f"stream_in/{name}.json", "w") as f:
        for m in minutes:
            f.write(json.dumps({"ts": f"2024-03-01 12:{m:02d}:00", "user": "u"}) + "\n")

def read_out():
    """The windows written so far, as [(window start 'HH:MM', count), ...] (given)."""
    if not os.path.isdir("stream_out") or not any(f.endswith(".parquet") for f in os.listdir("stream_out")):
        return []
    rows = (spark.read.parquet("stream_out")
            .select(F.date_format(F.col("window.start"), "HH:mm").alias("start"), "count").orderBy("start").collect())
    return [(r["start"], r["count"]) for r in rows]

for folder in ("stream_in", "stream_out", "stream_ckpt"):
    shutil.rmtree(folder, ignore_errors=True)

def run():
    events = (spark.readStream.schema("ts timestamp, user string").json("stream_in")
              .withWatermark("ts", "10 minutes")
              .groupBy(F.window("ts", "5 minutes")).count())
    q = (events.writeStream.outputMode("append").format("parquet").option("path", "stream_out")
         .option("checkpointLocation", "stream_ckpt").trigger(availableNow=True).start())
    q.awaitTermination()

results = []
for name, minutes in [("b1", [0, 1, 6, 9]), ("b2", [31]), ("b3", [7, 22]), ("b4", [45])]:
    drop_events(name, minutes)
    run()
    results.append(read_out())
result = results
result

In [ ]:
check("8.8", result)

## E. Coding practice: arrays and text

Two problems with array functions (no `explode`, no UDF) and text cleaning. Write them from scratch.

**8.9 Work inside arrays without exploding.** For the customers 2 to 6, collect the *amount* of all their *paid* orders into an array *amounts* (`collect_list`). Then use higher-order functions on the array (`F.expr` with `filter` and `aggregate`): *big* = how many amounts are over 500, *total* = the sum of the amounts (2 decimals). *rows* = list of `(customer_id, big, total)` sorted by *customer_id*.

In [ ]:
arrays = (orders.filter((F.col("status") == "paid") & F.col("customer_id").between(2, 6))
          .groupBy("customer_id").agg(F.collect_list("amount").alias("amounts")))
out = arrays.select("customer_id",
                    F.expr("size(filter(amounts, x -> x > 500))").alias("big"),
                    F.expr("aggregate(amounts, cast(0 as double), (acc, x) -> acc + x)").alias("total")).orderBy("customer_id")
rows = [(r["customer_id"], r["big"], round(r["total"], 2)) for r in out.collect()]
rows

In [ ]:
check("8.9", rows)

**8.10 Clean phone numbers.** *messy* (given) has phone numbers in many formats. Add *digits* (only the digits of *raw*, `regexp_replace`), *national* (the last 9 digits of *digits*, `null` when there are fewer than 9 digits) and *valid* (True when *national* is not null). *cleaned* = list of `(raw, national, valid)` in the original order.

In [ ]:
messy = spark.createDataFrame([("+998 (90) 123-45-67",), ("998901234567",), ("90-123-45-67",), ("8 800 555 35 35",), ("12-34",)], "raw string")

cleaned_df = (messy.withColumn("digits", F.regexp_replace("raw", "[^0-9]", ""))
              .withColumn("national", F.when(F.length("digits") >= 9, F.substring("digits", -9, 9)))
              .withColumn("valid", F.col("national").isNotNull()))
cleaned = [(r["raw"], r["national"], r["valid"]) for r in cleaned_df.collect()]
cleaned

In [ ]:
check("8.10", cleaned)

---
Done? In the Theory part of Day 8, answer the red checklist topics aloud, with the timings and plans from this lab as evidence.